# 07 — Resources and Prompts

**What you'll learn**
- MCP isn't only tools. The spec defines **three** primitives: `tools`, `resources`, `prompts`.
- When to expose data as a **resource** instead of a tool.
- How **prompts** let the server ship reusable templated prompts to any host.
- The same FastMCP server can expose all three side-by-side.

## The three MCP primitives

| Primitive | Direction | Has side effects? | Examples |
|---|---|---|---|
| `tools` | host → server (action) | Yes — writes, calls, mutations | `create_contact`, `send_email`, `run_query` |
| `resources` | host → server (read) | No — pure read | A specific doc, a CRM row, a log file, a config |
| `prompts` | host → server (read) | No — returns a templated prompt | "Draft a follow-up email", "Summarize this contact" |

Rule of thumb:
- **Action that changes state → tool.**
- **Data the model should read → resource.**
- **A reusable prompt template that uses your domain data → prompt.**

People often shove everything into tools because tools are easiest. That's fine, but you lose two nice properties:
1. **Resources are cacheable and listable.** The host can scan them without invoking anything.
2. **Prompts are sharable.** Anyone using your server gets your best-practice prompts for free.

## Extended Mini server

We extend `MiniMCPServer` from notebook 02 with `resource()` and `prompt()` decorators. Same simulation idea — the real FastMCP API mirrors this shape almost exactly.

In [ ]:
from typing import Callable, Any

class MiniMCPServer:
    """MCP-style server covering all three primitives.

    - tools     : callable side-effects (create_contact, etc.)
    - resources : read-only items addressable by URI (a doc, a row, a file)
    - prompts   : reusable templated prompts the host can fetch and fill in
    """

    def __init__(self, name: str) -> None:
        self.name = name
        # tools
        self._tools: dict[str, Callable[..., Any]] = {}
        self._tool_descs: dict[str, str] = {}
        # resources: uri -> (description, reader_fn)
        self._resources: dict[str, tuple[str, Callable[[], Any]]] = {}
        # prompts: name -> (description, builder_fn)
        self._prompts: dict[str, tuple[str, Callable[..., list[dict]]]] = {}

    # ---- tools ----
    def tool(self, description: str = ""):
        def deco(fn):
            self._tools[fn.__name__] = fn
            self._tool_descs[fn.__name__] = description or (fn.__doc__ or "").strip()
            return fn
        return deco

    def list_tools(self) -> list[dict]:
        return [{"name": n, "description": self._tool_descs[n]} for n in self._tools]

    def call_tool(self, name: str, arguments: dict):
        if name not in self._tools:
            raise ValueError(f"unknown tool: {name}")
        return self._tools[name](**arguments)

    # ---- resources ----
    def resource(self, uri: str, description: str = ""):
        def deco(fn):
            self._resources[uri] = (description or (fn.__doc__ or "").strip(), fn)
            return fn
        return deco

    def list_resources(self) -> list[dict]:
        return [{"uri": u, "description": d} for u, (d, _) in self._resources.items()]

    def read_resource(self, uri: str) -> Any:
        if uri not in self._resources:
            raise ValueError(f"unknown resource: {uri}")
        _, reader = self._resources[uri]
        return reader()

    # ---- prompts ----
    def prompt(self, name: str, description: str = ""):
        def deco(fn):
            self._prompts[name] = (description or (fn.__doc__ or "").strip(), fn)
            return fn
        return deco

    def list_prompts(self) -> list[dict]:
        return [{"name": n, "description": d} for n, (d, _) in self._prompts.items()]

    def get_prompt(self, name: str, arguments: dict | None = None) -> list[dict]:
        if name not in self._prompts:
            raise ValueError(f"unknown prompt: {name}")
        _, builder = self._prompts[name]
        return builder(**(arguments or {}))

## Part 1 — Resources

A resource is something the model can **read** by URI. URIs are arbitrary; common conventions:

- `crm://contacts/contact_42`
- `file:///etc/hosts`
- `db://orders/last_30d`
- `doc://onboarding/v3`

The server decides what the URI means. The host doesn't need to understand the scheme — it just lists and reads.

In [ ]:
# Toy CRM state for the demo
CONTACTS = {
    "contact_1": {"id": "contact_1", "name": "Ada Lovelace", "email": "ada@ex.com",
                  "owner_id": "osc_101"},
    "contact_2": {"id": "contact_2", "name": "Alan Turing",  "email": "alan@ex.com",
                  "owner_id": "osc_102"},
}
OSC_TEAM = [
    {"id": "osc_101", "name": "Ava OSC"},
    {"id": "osc_102", "name": "Ben OSC"},
    {"id": "osc_103", "name": "Cara OSC"},
]
RECENT_TASKS_DOC = """# Recent follow-up tasks

- Ada Lovelace — needs demo scheduled.
- Alan Turing — pricing question, send v2 deck.
- Grace Hopper — overdue follow-up from last week.
"""

server = MiniMCPServer("sales-server")


@server.resource("crm://team/osc-roster",
                 description="Current OSC team roster (read-only)")
def osc_roster():
    return OSC_TEAM


@server.resource("crm://contacts/all",
                 description="All contacts in the CRM")
def all_contacts():
    return list(CONTACTS.values())


@server.resource("doc://tasks/recent",
                 description="Markdown summary of recent follow-up tasks")
def recent_tasks():
    return RECENT_TASKS_DOC


server.list_resources()

### Reading a resource

A host calls `read_resource(uri)`. The server returns the bytes/JSON/text. The LLM treats it just like any other context — but it came from a discoverable, server-defined location, not from random `print` calls in your agent.

In [ ]:
print(server.read_resource("crm://contacts/all"))
print("---")
print(server.read_resource("doc://tasks/recent"))

### The real FastMCP equivalent

The decorator API is the same. The argument to `@mcp.resource()` is the URI pattern; you can use `{placeholder}` syntax for parameterized URIs like `crm://contacts/{contact_id}`.

In [ ]:
REAL_RESOURCE_SERVER = r'''# sales_mcp_resources.py
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("sales-with-resources")

CONTACTS = {"contact_1": {"id": "contact_1", "name": "Ada", "email": "a@ex.com"}}


@mcp.resource("crm://contacts/all")
def all_contacts() -> list[dict]:
    """All contacts in the CRM."""
    return list(CONTACTS.values())


@mcp.resource("crm://contacts/{contact_id}")
def one_contact(contact_id: str) -> dict:
    """A specific contact by id."""
    return CONTACTS[contact_id]


if __name__ == "__main__":
    mcp.run()
'''
print(REAL_RESOURCE_SERVER)

## Part 2 — Prompts

A prompt is a **named, parameterized message list** the server hands back to the host. The host typically shows these in a UI ("Use the `follow_up_email` prompt") and the LLM then runs with that filled-in prompt.

Why this matters: your server is the one that knows the domain. If you write a great "summarize this contact" prompt once, every host that connects to your server gets it. No more copy-pasting prompt text between teammates.

In [ ]:
@server.prompt("contact_summary",
                description="Summarize a contact in 3 bullets for a sales rep")
def contact_summary_prompt(contact_id: str):
    contact = CONTACTS[contact_id]
    return [
        {"role": "system",
         "content": "You are a senior account executive. Be terse, signal-only."},
        {"role": "user",
         "content": f"Summarize this contact in 3 bullets:\n{contact}"},
    ]


@server.prompt("follow_up_email",
                description="Draft a short, friendly follow-up email")
def follow_up_email_prompt(contact_id: str, topic: str):
    contact = CONTACTS[contact_id]
    return [
        {"role": "system",
         "content": "You write short, warm follow-up emails — 4 sentences max."},
        {"role": "user",
         "content": (f"Write a follow-up email to {contact['name']} "
                     f"({contact['email']}) about: {topic}")},
    ]


server.list_prompts()

### Getting a prompt back filled-in

In [ ]:
messages = server.get_prompt("contact_summary", {"contact_id": "contact_1"})
for m in messages:
    print(m["role"].upper(), ":", m["content"])
print()

messages = server.get_prompt("follow_up_email",
                             {"contact_id": "contact_2",
                              "topic": "pricing for the v2 plan"})
for m in messages:
    print(m["role"].upper(), ":", m["content"])

### The real FastMCP equivalent

In [ ]:
REAL_PROMPT_SERVER = r'''# sales_mcp_prompts.py
from mcp.server.fastmcp import FastMCP
from mcp.server.fastmcp.prompts import base

mcp = FastMCP("sales-with-prompts")


@mcp.prompt()
def contact_summary(contact_id: str) -> list[base.Message]:
    """Summarize a contact in 3 bullets for a sales rep."""
    contact = lookup_contact(contact_id)
    return [
        base.SystemMessage("You are a senior AE. Be terse, signal-only."),
        base.UserMessage(f"Summarize this contact in 3 bullets:\n{contact}"),
    ]


@mcp.prompt()
def follow_up_email(contact_id: str, topic: str) -> list[base.Message]:
    """Draft a short, friendly follow-up email."""
    contact = lookup_contact(contact_id)
    return [
        base.SystemMessage("You write short, warm follow-up emails — 4 sentences max."),
        base.UserMessage(
            f"Write a follow-up to {contact['name']} ({contact['email']}) about: {topic}"
        ),
    ]


if __name__ == "__main__":
    mcp.run()
'''
print(REAL_PROMPT_SERVER)

## Putting all three together

One server can expose tools, resources, and prompts at the same time. That is the typical shape for a real MCP server.

In [ ]:
@server.tool(description="Add a quick note to a contact")
def add_note(contact_id: str, note: str) -> dict:
    contact = CONTACTS[contact_id]
    contact.setdefault("notes", []).append(note)
    return contact


print("TOOLS:    ", [t["name"] for t in server.list_tools()])
print("RESOURCES:", [r["uri"]  for r in server.list_resources()])
print("PROMPTS:  ", [p["name"] for p in server.list_prompts()])

## Mini test

In [ ]:
# Tools work
result = server.call_tool("add_note",
                          {"contact_id": "contact_1", "note": "Likes haskell jokes"})
assert "Likes haskell jokes" in result["notes"]

# Resources work
roster = server.read_resource("crm://team/osc-roster")
assert any(o["id"] == "osc_101" for o in roster)

# Prompts work
msgs = server.get_prompt("contact_summary", {"contact_id": "contact_1"})
assert msgs[0]["role"] == "system"
assert "Ada Lovelace" in msgs[1]["content"]

# Unknown items raise
for fn, arg in [(server.call_tool,    ("nope", {})),
                (server.read_resource, ("crm://nope",)),
                (server.get_prompt,    ("nope", None))]:
    try:
        fn(*arg)
    except ValueError as e:
        pass
    else:
        raise AssertionError(f"{fn.__name__} should have raised")

print("ok")

## Key takeaway

If you only know `@mcp.tool()`, you have one third of the protocol. **Tools** are for actions, **resources** are for data, **prompts** are for reusable templates. A well-designed MCP server uses all three so the host (and the human behind it) gets data, capabilities, and best-practice prompts in one package.